# 02 · Sequences: padding, masks, LSTMs, pooling (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/02_sequence_models.md`.

**Traces to** `notes/ioai-task-patterns.md` P9. T25-G-WORDSEG gives a char vocabulary with **0 reserved for
padding**, a `collate_fn` that pads and returns lengths, and a **mask applied before the BCE loss**. It recommended
"Embedding + LSTM". The same padding/masking discipline returns in every HF tokenizer call (`attention_mask`).

Task: split concatenated English words ("footballplayer" → "football|player"), labelling each character 1 if
a word ends there. Runtime: about 1.5 minutes on a T4.

In [ ]:
import re, collections, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset

torch.manual_seed(0); random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

texts = load_dataset("fancyzhx/ag_news", split="train")["text"][:40000]
cnt = collections.Counter(w for t in texts for w in re.findall(r"[a-z]+", t.lower()))
WORDS = [w for w, _ in cnt.most_common(6000) if 3 <= len(w) <= 8][:2500]

def make_compounds(n, rng):
    out = {}
    while len(out) < n:
        parts = [WORDS[i] for i in rng.integers(0, len(WORDS), rng.integers(2, 5))]
        out["".join(parts)] = [b for p in parts for b in [0] * (len(p) - 1) + [1]]
    return out

rng = np.random.default_rng(0)
train_data = list(make_compounds(30000, rng).items())
val_data = list(make_compounds(3000, rng).items())
print(train_data[0])

## 1. Vocabulary with a reserved padding id

In [ ]:
chars = sorted({c for w, _ in train_data for c in w})
# TODO(g1): map each char to an id, starting at 1, so that 0 is free for padding
char2idx = ____
VOCAB = len(char2idx) + 1                     # +1 for the padding id 0
encode = lambda w: torch.tensor([char2idx.get(c, 0) for c in w], dtype=torch.long)

In [ ]:
assert 0 not in char2idx.values() and sorted(char2idx.values()) == list(range(1, len(chars) + 1))
print("g1 ok, vocab size", VOCAB)

## 2. Padding in `collate_fn`

In [ ]:
def collate(batch):
    seqs = [encode(w) for w, _ in batch]
    labs = [torch.tensor(l, dtype=torch.float) for _, l in batch]
    lengths = torch.tensor([len(s) for s in seqs])
    # TODO(g2): pad the id sequences into one (B, L_max) tensor, batch first, with the padding id
    ids = pad_sequence(seqs, ____)
    y = pad_sequence(labs, batch_first=True, padding_value=0.0)
    return ids, y, lengths

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_data, batch_size=512, collate_fn=collate)

In [ ]:
ids, y, lengths = collate([("abcde", [0, 0, 1, 0, 1]), ("xy", [0, 1])])
assert ids.shape == (2, 5) and ids[1, 2:].tolist() == [0, 0, 0] and y.dtype == torch.float32
print("g2 ok")

## 3. Embedding + BiLSTM tagger

Packing (`pack_padded_sequence`) makes the LSTM skip padded steps. Without it the **backward** direction starts
from the pads, and a word's outputs change depending on which longer word shares its batch.

In [ ]:
class Tagger(nn.Module):
    def __init__(self, vocab, emb=32, hidden=96):
        super().__init__()
        # TODO(g3): embedding whose padding row stays zero and gets no gradient
        self.emb = nn.Embedding(vocab, emb, ____)
        self.lstm = nn.LSTM(emb, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        # TODO(g4): a bidirectional LSTM concatenates forward and backward states: what is the feature size?
        self.out = nn.Linear(____, 1)

    def forward(self, ids, lengths):
        x = self.emb(ids)
        # TODO(g5): pack using the true lengths (lengths must live on the CPU; batch is not sorted)
        packed = ____
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.shape[1])
        return self.out(h).squeeze(-1), h           # logits (B, L), states (B, L, 2H)

model = Tagger(VOCAB).to(device)

In [ ]:
model.eval()
a = encode("cat").unsqueeze(0).to(device)
alone, _ = model(a, torch.tensor([3]))
both_ids, _, both_len = collate([("cat", [0, 0, 1]), ("elephantine", [0] * 10 + [1])])
together, _ = model(both_ids.to(device), both_len)
assert model.emb.padding_idx == 0 and model.out.in_features == 192
assert torch.allclose(alone[0], together[0, :3], atol=2e-3), "padding leaked into the outputs: pack the sequence"
print("g3, g4, g5 ok")

## 4. Masked loss

Padded positions have target 0 and would teach the model to predict "no boundary" on garbage.

In [ ]:
bce = nn.BCEWithLogitsLoss(reduction="none")

def masked_loss(logits, y, lengths):
    L = logits.shape[1]
    # TODO(g6): boolean (B, L) mask, True where the position is a real character
    mask = ____
    # TODO(g7): average the per-position loss over REAL positions only
    return ____

In [ ]:
lg = torch.zeros(2, 4); yy = torch.tensor([[1., 0, 0, 0], [1, 1, 1, 1]]); ln = torch.tensor([1, 4])
assert torch.isclose(masked_loss(lg, yy, ln), torch.tensor(np.log(2), dtype=torch.float32))
lg2 = lg.clone(); lg2[0, 1:] = 100.0                        # garbage at padded positions must not matter
assert torch.isclose(masked_loss(lg2, yy, ln), masked_loss(lg, yy, ln))
print("g6, g7 ok")

## 5. Train and score with the official T25-G-WORDSEG metric (segment F1)

In [ ]:
def segments(labels):
    segs, start = set(), 0
    for i, v in enumerate(labels):
        if v == 1: segs.add((start, i + 1)); start = i + 1
    if start < len(labels): segs.add((start, len(labels)))
    return segs

def word_f1(gold, pred):
    g, p = segments(gold), segments(pred)
    tp = len(g & p); pr = tp / len(p) if p else 0; rc = tp / len(g) if g else 0
    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); scores = []
    for ids, y, lengths in loader:
        logits, _ = model(ids.to(device), lengths)
        pred = (logits > 0).int().cpu()
        for i, n in enumerate(lengths.tolist()):
            p = pred[i, :n].tolist(); p[-1] = 1              # the last char always ends a word
            scores.append(word_f1(y[i, :n].int().tolist(), p))
    return float(np.mean(scores))

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for epoch in range(3):
    model.train()
    for ids, y, lengths in train_loader:
        logits, _ = model(ids.to(device), lengths)
        loss = masked_loss(logits, y.to(device), lengths)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    print(f"epoch {epoch}  loss {loss.item():.4f}  val segment-F1 {evaluate(model, val_loader):.4f}")

## 6. Pooling a whole sequence into one vector

For sequence-level outputs (T26-FIND's clip embeddings, sentence embeddings) we pool over time. Pads must be
excluded from the mean, and from the max via a `-inf` fill.

In [ ]:
@torch.no_grad()
def pooled(model, words):
    ids, _, lengths = collate([(w, [0] * len(w)) for w in words])
    _, h = model(ids.to(device), lengths)                                 # (B, L, D)
    mask = (torch.arange(h.shape[1], device=device)[None] < lengths.to(device)[:, None]).unsqueeze(-1)  # (B, L, 1)
    # TODO(g8): masked mean over time: sum real positions, divide by the true length
    mean = ____
    mx = h.masked_fill(~mask, float("-inf")).max(1).values
    return mean, mx

m1, x1 = pooled(model, ["footballplayer"])
m2, x2 = pooled(model, ["footballplayer", "abc"])
m3, x3 = pooled(model, ["footballplayer", "nationalbasketballassociation"])

In [ ]:
assert torch.allclose(m1[0], m2[0], atol=2e-3) and torch.allclose(m1[0], m3[0], atol=2e-3), "mean depends on padding"
assert torch.allclose(x1[0], x3[0], atol=2e-3)
print("g8 ok, all gaps done")